# Operations Research 3: Examples for Theory

This code is for students who take the class Operations Research. Students should finish the installation of Gurobi and Python before workersed and make sure an academic liscense for Gurobi is applied and activated.

We introduce two examples for shadow price in order to help students understand how to implement theories introduced in lectures with codes. More instruction is provided in the lecture video.|

# Shadow Price
## Example 1: Producing desks and tables


Consider the problem we introduced in Operations Research: Modeling and Application, we have

\begin{split}
    \begin{array}{r}
        \max \\ \mbox{s.t.} \\ \\ \\ \\ \\
    \end{array} &
    \begin{array}{rcrcll}
        700x_1 & + & 900x_2 & & & \\
        3x_1 & + & 5x_2 & \leq & 3600\quad\! & \mbox{(wood)} \\
        x_1 & + & 2x_2 & \leq & 1600\quad\! & \mbox{(labor)} \\
        50x_1 & + & 20x_2 & \leq & 48000\:\ & \mbox{(machine)} \\
        x_1 & & & \geq & 0
        \\
        & & x_2 & \geq & 0.
    \end{array}
\end{split}


We use the code introduced in Operations Research: Algorithms to show shadow prices Gurobi.\
Let's construct our problem step by step.

In [4]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

In [29]:
# change the color and font style when printing string

class color:
   PURPLE = '\033[95m'
   CYAN = '\033[96m'
   DARKCYAN = '\033[36m'
   BLUE = '\033[94m'
   GREEN = '\033[92m'
   YELLOW = '\033[93m'
   RED = '\033[91m'
   BOLD = '\033[1m'
   UNDERLINE = '\033[4m'
   END = '\033[0m'

Different from the code we used before, we put our data and model in a fuction to make adjust the resource limitation (the RHS of constraint) more easily.

In [65]:
def linear_example(limitation_list):

    """Data Part"""

    prices = np.array([700, 900])  # Matrix c
    resource_consumptions = np.array([[3 , 5 ],
                             [1 , 2 ],
                             [50, 20]])  # Matrix A
    resource_limitations = np.array(limitation_list) # modify by function inputs, Matrix b


    """Model Part"""
    # with gp.Model("LP_example") as eg1:  # Model is deleted when reaching the return statement
    eg1 = gp.Model("LP_example")

    # 1. Add matrix variables
    variable_names = [f"x{i+1}" for i in range(len(prices))]
    x = eg1.addMVar(shape = len(prices), lb = 0, name= variable_names)

    # 2. Add matrix constraints
    eg1.addMConstr(resource_consumptions, x, "<=", resource_limitations, name = 'resource_limitations')

    # 3. Set Objective
    eg1.setObjective(prices @ x, GRB.MAXIMIZE)

    # Step 1 to 3 in a loop manner
    # x = []
    # for i in products:
    #     x.append(eg1.addVar(lb = 0, vtype = GRB.CONTINUOUS, name = 'x' + str(i)))
    #
    # eg1.setObjective(quicksum(prices[i] * x[i] for i in products) , GRB.MAXIMIZE)
    #
    # # add constraints and name them
    # eg1.addConstrs((quicksum(resource_consumptions[j][i] * x[i] for i in products)
    #                            <= resource_limitations[j] for j in resources), "Resource_limitation")

    # 4. Call the solver
    eg1.optimize()

    # Check model status and access solution
    if eg1.Status == GRB.OPTIMAL:
        for var in eg1.getVars():
            print(var.varName, '=', round(var.x, 2))
        print("objective value =", round(eg1.objVal, 2))
    else:
        print("Optimal solution not found")

    return eg1  # Return the model for accessing the shadow prices

In [66]:
limitation_list = np.array([3600, 1600, 48000])
origin_model1 = linear_example(limitation_list)

# use Pi to get the shadow price of each constraint
print('The shadow price of the wood constraint is {shadow_price}.'.format(shadow_price = origin_model1.Pi[0]))
print('The shadow price of the labor constraint is {shadow_price}.'.format(shadow_price = origin_model1.Pi[1]))
print('The shadow price of the machine constraint is {shadow_price}.'.format(shadow_price = origin_model1.Pi[2]))

Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 5950X 16-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 32 logical processors, using up to 32 threads

Optimize a model with 3 rows, 2 columns and 6 nonzeros (Max)
Model fingerprint: 0xfc16a329
Model has 2 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [7e+02, 9e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+03, 5e+04]
Presolve time: 0.00s
Presolved: 3 rows, 2 columns, 6 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.0000000e+32   3.593750e+30   2.000000e+02      0s
       3    7.8947368e+05   0.000000e+00   0.000000e+00      0s

Solved in 3 iterations and 0.01 seconds (0.00 work units)
Optimal objective  7.894736842e+05
x1 = 884.21
x2 = 189.47
objective value = 789473.68
The shadow price of the wood constraint is 163.1578947368421.


Use a loop to adjust the three constraints in our model.

In [67]:
for i in range(3):
    print('==============================================')

    # print the shadow prices of constraints and review the expected objective value
    print(color.DARKCYAN + color.BOLD +
          'Shadow price of {name} is {shadow_price}'.format(name = origin_model1.ConstrName[i], shadow_price = round(origin_model1.Pi[i], 2))
          + color.END)

    # print the original objective value
    print(color.DARKCYAN + color.BOLD +
          'Original objective value: {obj}'.format(obj = round(origin_model1.objVal, 2)) + color.END)

    # set new RHS
    new_limitation_list = limitation_list.copy()
    new_limitation_list[i] = limitation_list[i] + 1

    # construct the model with new constraints
    new_model1 = linear_example(new_limitation_list)
    print(color.DARKCYAN + color.BOLD +
          'Objective value after adding {name} by one unit: {obj}'.format(name = new_model1.ConstrName[i],obj = round(new_model1.objVal, 2))
          + color.END)

Shadow price of resource_limitations[0] is 163.16
Original objective value: 789473.68
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 5950X 16-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 32 logical processors, using up to 32 threads

Optimize a model with 3 rows, 2 columns and 6 nonzeros (Max)
Model fingerprint: 0x83b7242e
Model has 2 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [7e+02, 9e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+03, 5e+04]
Presolve time: 0.00s
Presolved: 3 rows, 2 columns, 6 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.0000000e+32   3.593750e+30   2.000000e+02      0s
       3    7.8963684e+05   0.000000e+00   0.000000e+00      0s

Solved in 3 iterations and 0.01 seconds (0.00 work units)
Optimal objective  7.896368421e+05
x1 = 884.11
x2 = 189.74
obje

## Example 2: Personnel Scheduling

Consider the personnel scheduling problem we introduced in Operations Research: Modeling and Application, we have

\begin{split}
    \begin{array}{rrllllllllllllll}
        \min \ & x_1 & + & x_2 & + & x_3 & + & x_4 & + & x_5 & + & x_6 & + & x_7  \\
        \mbox{s.t.} \ & x_1 & + &&&&& x_4 & + & x_5 & + & x_6 & + & x_7 & \geq 110\\
        & x_1 & + & x_2 & + &&&&& x_5 & + & x_6 & + & x_7 & \geq 80\\
        & x_1 & + & x_2 & + & x_3 & + &&&&& x_6 & + & x_7 & \geq 150\\
        & x_1 & + & x_2 & + & x_3 & + & x_4 & + &&&&& x_7 & \geq 30\\
        & x_1 & + & x_2 & + & x_3 & + & x_4 & + & x_5 &&&&& \geq 70\\
        &&& x_2 & + & x_3 & + & x_4 & + & x_5 & + & x_6 &&& \geq 160\\
        &&&&& x_3 & + & x_4 & + & x_5 & + & x_6 & + & x_7 & \geq 120\\
    \end{array} \\
    x_i \geq 0 \quad \forall i = 1,...,7.\hspace{9.1cm}
\end{split}

We also used the code intoduced in Operations Research: Algorithms.
Let's construct the problem and check the shadow prices of the model.

In [68]:
import pandas as pd

First, we import personnel scheduling constraints which is limitations of people needed in each day. More details were mentioned in Operations Research: Modeling and Application.

In [69]:
personnel_con = pd.read_excel('personnel_scheduling.xlsx', 'people_needed')
print(personnel_con)

  weekday  people_needed
0     Mon            110
1     Tue             80
2     Wed            150
3     Thu             30
4     Fri             70
5     Sat            160
6     Sun            120


Same as th last example, we put the code of data and model part into a function.

In [1]:
# daily working people w.r.t shift plans: Indices in Matrix
# For example, shift works through Monday to Friday of 5 work days
num_of_days = 7
shifts = 7
work_days = 5

A = np.zeros((num_of_days, num_of_days))
for s in range(shifts):
    if s + work_days <= num_of_days:
        A[s : s + work_days, s] = 1
    else:
        A[s:, s] = 1  # fill to the end of the week
        remaining_days = work_days - (num_of_days - s)
        A[:remaining_days, s] = 1  # 5 workdays - (from shift to sunday)

print(A)

NameError: name 'np' is not defined

In [71]:
def personnel_scheduling_example(constraints=personnel_con['people_needed']):
    """Data Part"""

    num_of_days = 7
    shifts = 7
    work_days = 5

    A = np.zeros((num_of_days, num_of_days))  # Matrix A
    for s in range(shifts):
        if s + work_days <= num_of_days:
            A[s : s + work_days, s] = 1
        else:
            A[s:, s] = 1  # fill to the end of the week
            remaining_days = work_days - (num_of_days - s)
            A[:remaining_days, s] = 1  # 5 workdays - (from shift to sunday)

    day_limitations = constraints  # Matrix b
    c = np.ones(num_of_days)  # Matrix c

    """Model Part"""

    eg2 = gp.Model("personnel_scheduling")

    # 1. Add matrix variables
    variable_names = [f"shift{i+1}" for i in range(len(day_limitations))]
    x = eg2.addMVar(shape = len(constraints), lb = 0, vtype=GRB.CONTINUOUS, name= variable_names)

    # In personnel scheduling, you usually can't hire "0.5 people." If the problem requires whole people, you should change your variable definition to include
    # vtype = GRB.INTEGER
    # If you use Integer variables, you cannot access shadow prices (.Pi), as duals are only defined for continuous problems.

    # 2. Add matrix constraints
    eg2.addMConstr(A, x, ">=", day_limitations, name = 'daily_people_limitations')

    # 3. Set Objective
    eg2.setObjective(c @ x, GRB.MINIMIZE)

    # # add variables
    # x = []
    # for i in people:
    #     x.append(eg2.addVar(lb=0, vtype=GRB.CONTINUOUS, name='x' + str(i)))
    #
    # # set objective value
    # eg2.setObjective(quicksum(x[i] for i in people), GRB.MINIMIZE)
    #
    # # add constraints and name them
    # eg2.addConstrs((quicksum(x[i + j - 7] for i in range(3, 8))
    #                 >= day_limitations[j] for j in days), "Resource_limitation")

     # 4. Call the solver
    eg2.optimize()

    # Check model status and access solution
    if eg2.Status == GRB.OPTIMAL:
        for var in eg2.getVars():
            print(var.varName, '=', round(var.x, 2))
        print("objective value =", round(eg2.objVal, 2))
    else:
        print("Optimal solution not found")

    return eg2  # Return the model for accessing the shadow prices

In [72]:
origin_model2 = personnel_scheduling_example()

Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 5950X 16-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 32 logical processors, using up to 32 threads

Optimize a model with 7 rows, 7 columns and 35 nonzeros (Min)
Model fingerprint: 0xa040a86d
Model has 7 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+01, 2e+02]
Presolve time: 0.00s
Presolved: 7 rows, 7 columns, 35 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   7.200000e+02   0.000000e+00      0s
       5    1.6333333e+02   0.000000e+00   0.000000e+00      0s

Solved in 5 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.633333333e+02
shift1 = 3.33
shift2 = 40.0
shift3 = 13.33
shift4 = 13.33
shift5 = 0.0
shift6 = 93.33
shift7 = 0.0
objective val

Use loop to view the shadow prices of each constraints. We check the shadow prices and the expected objective value, and then compare the different between expected and actual objective value.

In [73]:
for i in range(7):
    print('==============================================')

    # print the shadow prices of constraints and review the expected objective value
    print(color.DARKCYAN + color.BOLD +
          'Shadow price of {name} is {shadow_price}'.format(name=origin_model2.ConstrName[i],
                                                            shadow_price=round(origin_model2.Pi[i], 2))
          + color.END)
    print(color.DARKCYAN + color.BOLD + 'Original objective value: {obj}'.format(
        obj=round(origin_model2.objVal, 2)) + color.END)

    # set new RHS
    new_days_limitation = personnel_con['people_needed'].copy()
    new_days_limitation[i] = new_days_limitation[i] + 1
    new_model2 = personnel_scheduling_example(new_days_limitation)
    print(color.DARKCYAN + color.BOLD +
          'Objective value after adding {name} one unit: {obj}'.format(name=new_model2.ConstrName[i],
                                                                       obj=round(new_model2.objVal, 2))
          + color.END)

Shadow price of daily_people_limitations[0] is 0.33
Original objective value: 163.33
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 5950X 16-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 32 logical processors, using up to 32 threads

Optimize a model with 7 rows, 7 columns and 35 nonzeros (Min)
Model fingerprint: 0xa73572d6
Model has 7 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+01, 2e+02]
Presolve time: 0.00s
Presolved: 7 rows, 7 columns, 35 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   7.210000e+02   0.000000e+00      0s
       5    1.6366667e+02   0.000000e+00   0.000000e+00      0s

Solved in 5 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.636666667e+02
shift1 = 3.67
shift2 = 40.0